In [1]:
## Load OS module
import os

## Add environment variable for R installation
#os.environ['R_HOME'] = r"C:\Program Files\R\R-4.4.1"
os.environ["PATH"] = r"C:\Users\ArPa3547\AppData\Local\Programs\R\R-4.6.0\bin\x64"

In [ ]:
from healthiar.healthiar import healthiar
from healthiar.conversion import py_to_r, r_to_py

geo_id_micro = py_to_r(["Zurich", "Basel", "Geneva", "Ticino", "Jura"])
geo_id_macro = py_to_r(["German","German","French","Italian","French"])
exp_central = py_to_r([11, 11, 10, 8, 7])
bhd_central = py_to_r([4000, 2500, 3000, 1500, 500])

results_iteration = healthiar.attribute_health(
# Names of Swiss cantons
    geo_id_micro = geo_id_micro,
    # Names of languages spoken in the selected Swiss cantons
    geo_id_macro = geo_id_macro,
    rr_central = 1.369,
    rr_increment = 10, 
    cutoff_central = 5,
    erf_shape = "log_linear",
    exp_central = exp_central,
    bhd_central = bhd_central
)

py_results_iteration = r_to_py(results_iteration)
py_results_iteration["health_main"][["geo_id_macro", "impact_rounded", "erf_ci", "exp_ci", "bhd_ci"]].head()

In [ ]:
import rpy2.rinterface as ri
from scipy.interpolate import CubicSpline

## define ERF with decorator ('@')
@ri.rternalize
def erf_fun(x):
    cs = CubicSpline(
      x = [0, 5, 10, 15, 20, 25, 30, 50, 70, 90, 110],
      y = [1.00, 1.04, 1.08, 1.12, 1.16, 1.20, 1.23, 1.35, 1.45, 1.53, 1.60]
    )
    return float(cs(x)[0])

## pass ERF to attribute_health()
results_pm_copd_mr_brt = healthiar.attribute_health(
  exp_central = 8.85,
  bhd_central = 30747,
  cutoff_central = 0,
  erf_eq_central = erf_fun
)

In [ ]:
from healthiar.spatial import read_raster, read_vector
from importlib.resources import files # to access package data files
import pathlib # to convert Windows path to POSIX path, if running Windows

path = files("healthiar.data")
exdat_pwm_1 = read_raster(path.joinpath("pm25.tif").as_posix())
exdat_pwm_2 = read_vector(path.joinpath("municipalities_brussels.gpkg").as_posix(), quiet = True)
print(type(exdat_pwm_2))

population = [27335, 131426, 86534, 25425, 59840, 42638, 125734, 35369, 203105, 50060, 44735, 57993, 89204, 54077, 22901, 99096, 25630, 49704, 25441]

geo_id_macro = ["North", "North", "South", "South", "East", "East", "West", "East", "Center", "East", "North", "South", "South", "North", "West", "West", "West", "South", "West"]

pwm = healthiar.prepare_exposure(
  poll_grid = exdat_pwm_1, # Formal class SpatRaster,
  geo_units = exdat_pwm_2, # sf of the geographic sub-units
  population = py_to_r(population), # population per geographic sub-unit
  geo_id_macro = py_to_r(geo_id_macro)
)

#py_pwm = r_to_py(pwm)
#print(py_pwm["exposure_main"])

In [ ]:
#from healthiar.spatial import read_raster, read_vector
from importlib.resources import files # to access package data files
import pathlib # to convert Windows path to POSIX path, if running Windows
import os

from rpy2.robjects.packages import importr

import geopandas as gpd

import healthiar

## Import R packages
sf = importr("sf")

path = files("healthiar.data")
obj = gpd.read_file(path.joinpath("municipalities_brussels.gpkg"))
obj.head()
#obj.plot(facecolor = "none")
print(type(obj))
print(isinstance(obj, gpd.geodataframe.GeoDataFrame))

obj.to_file(".temp_conv_vector.gpkg", layer = "1", driver = "GPKG")
obj = sf.st_read(".temp_conv_vector.gpkg", quiet = True)
print(type(obj))

os.remove(".temp_conv_vector.gpkg")

#obj.plot(facecolor = "none")



In [ ]:
#from healthiar.spatial import read_raster, read_vector
from importlib.resources import files # to access package data files
import pathlib # to convert Windows path to POSIX path, if running Windows

from rpy2.robjects.packages import importr

import os
import xarray as xr
import rioxarray
#import rasterio

## Import R packages
terra = importr("terra")

path = files("healthiar.data")
obj = xr.open_dataset(path.joinpath("pm25.tif"), engine = "rasterio", masked = True)
print(type(obj))
print(isinstance(obj, xr.core.dataset.Dataset))
#print(obj.dims)
#obj = obj.squeeze()
#print(obj.dims)
#print(type(obj))
obj.squeeze().rio.to_raster(".temp_conv_raster.tiff")
obj = terra.rast(".temp_conv_raster.tiff")
os.remove(".temp_conv_raster.tiff")
print(type(obj))


In [ ]:
from healthiar.healthiar import healthiar
from healthiar.conversion import py_to_r, r_to_py
from rpy2.robjects.packages import importr
from importlib.resources import files # to access package data files
import pathlib # to convert Windows path to POSIX path, if running Windows

sf = importr("sf")
terra = importr("terra")

path = files("healthiar.data")
exdat_pwm_1 = terra.rast(path.joinpath("pm25.tif").as_posix())
exdat_pwm_2 = sf.st_read(path.joinpath("municipalities_brussels.gpkg").as_posix(), quiet = True)

population = [27335, 131426, 86534, 25425, 59840, 42638, 125734, 35369, 203105, 50060, 44735, 57993, 89204, 54077, 22901, 99096, 25630, 49704, 25441]
geo_id_macro = ["North", "North", "South", "South", "East", "East", "West", "East", "Center", "East", "North", "South", "South", "North", "West", "West", "West", "South", "West"]

pwm = healthiar.prepare_exposure(
  poll_grid = exdat_pwm_1, # Formal class SpatRaster,
  geo_units = exdat_pwm_2, # sf of the geographic sub-units
  population = py_to_r(population), # population per geographic sub-unit
  geo_id_macro = py_to_r(geo_id_macro)
)

py_pwm = r_to_py(pwm)
print(py_pwm["exposure_main"])

In [ ]:
## Import modules
from healthiar.healthiar import healthiar
from healthiar.conversion import py_to_r, r_to_py
import os
from importlib.resources import files # to access package data files
import pathlib # to convert Windows path to POSIX path, if running Windows
import tempfile

import geopandas as gpd
import xarray as xr
import rioxarray

path = files("healthiar.data")

exdat_pwm_1 = xr.open_dataset(path.joinpath("pm25.tif"), engine = "rasterio", masked = True)
tmp_dir = pathlib.Path(tempfile.mkdtemp())
#tmp_dir = tempfile.mkdtemp()
print(tmp_dir)

exdat_pwm_1.squeeze().rio.to_raster(tmp_dir.joinpath(".temp_conv_raster.tiff"))
exdat_pwm_1 = terra.rast(tmp_dir.joinpath(".temp_conv_raster.tiff").as_posix())

exdat_pwm_2 = gpd.read_file(path.joinpath("municipalities_brussels.gpkg"))
tmp_dir = pathlib.Path(tempfile.mkdtemp())
print(tmp_dir)

exdat_pwm_2.to_file(tmp_dir.joinpath(".temp_conv_vector.gpkg"), layer = "1", driver = "GPKG")
exdat_pwm_2 = sf.st_read(tmp_dir.joinpath(".temp_conv_vector.gpkg").as_posix(), quiet = True)

"""
population = [27335, 131426, 86534, 25425, 59840, 42638, 125734, 35369, 203105, 50060, 44735, 57993, 89204, 54077, 22901, 99096, 25630, 49704, 25441]
geo_id_macro = ["North", "North", "South", "South", "East", "East", "West", "East", "Center", "East", "North", "South", "South", "North", "West", "West", "West", "South", "West"]

pwm = healthiar.prepare_exposure(
  poll_grid = exdat_pwm_1, # Formal class SpatRaster,
  geo_units = exdat_pwm_2, # sf of the geographic sub-units
  population = py_to_r(population), # population per geographic sub-unit
  geo_id_macro = py_to_r(geo_id_macro)
)

py_pwm = r_to_py(pwm)
print(py_pwm["exposure_main"])
"""



In [ ]:
## Import modules
from healthiar.healthiar import healthiar
from healthiar.conversion import py_to_r, r_to_py
import os
from importlib.resources import files # to access package data files
import pathlib # to convert Windows path to POSIX path, if running Windows



path = files("healthiar.data")
exdat_pwm_1 = xr.open_dataset(path.joinpath("pm25.tif"), engine = "rasterio", masked = True)
exdat_pwm_1 = py_to_r(exdat_pwm_1)
exdat_pwm_2 = gpd.read_file(path.joinpath("municipalities_brussels.gpkg"))
exdat_pwm_2 = py_to_r(exdat_pwm_2)

population = [27335, 131426, 86534, 25425, 59840, 42638, 125734, 35369, 203105, 50060, 44735, 57993, 89204, 54077, 22901, 99096, 25630, 49704, 25441]
geo_id_macro = ["North", "North", "South", "South", "East", "East", "West", "East", "Center", "East", "North", "South", "South", "North", "West", "West", "West", "South", "West"]

pwm = healthiar.prepare_exposure(
  poll_grid = exdat_pwm_1, # Formal class SpatRaster,
  geo_units = exdat_pwm_2, # sf of the geographic sub-units
  population = py_to_r(population), # population per geographic sub-unit
  geo_id_macro = py_to_r(geo_id_macro)
)

py_pwm = r_to_py(pwm)
print(py_pwm["exposure_main"])



{'geo_id_macro': ['Center', 'East', 'North', 'South', 'West'], 'exposure_mean': [11.4727144241333, 11.107156789619966, 11.48669642453657, 11.099398284732528, 11.388491724385805], 'population_total': [203105, 187907, 257573, 308860, 298802]}


In [4]:
from healthiar.conversion import py_to_r

geo_units = gpd.read_file(path.joinpath("municipalities_brussels.gpkg"))
print(geo_units["name"])
py_to_r(geo_units["name"])

0           Sint-Joost-ten-Node
1                    Schaarbeek
2                         Ukkel
3           Watermaal-Bosvoorde
4        Sint-Lambrechts-Woluwe
5           Sint-Pieters-Woluwe
6                    Anderlecht
7                      Oudergem
8                       Brussel
9                     Etterbeek
10                        Evere
11    Vorst (Brussel-Hoofdstad)
12                       Elsene
13                        Jette
14                   Koekelberg
15          Sint-Jans-Molenbeek
16                    Ganshoren
17                  Sint-Gillis
18          Sint-Agatha-Berchem
Name: name, dtype: str


'Sint-Joo...,'Schaarbe...,'Ukkel',...,'Ganshoren','Sint-Gil...,'Sint-Aga...
